In [1]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [2]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


10

In [3]:
# =============================================================================
# BLOCK 3: K-FOLD TRAINING OF MEAN MODEL (NO TUNING)
# =============================================================================
print("\n--- STAGE 1: K-Fold Training of Mean Model ---")
print("# Using pre-tuned, optimal hyperparameters.")
# --- YOUR BEST PARAMETERS FOR THE MEAN MODEL ---
# These are the parameters from your most successful Optuna run.
best_params_mean = {
            'eta': 0.041599605162930035,
            'max_depth': 8,
            'subsample': 0.8211034324219306,
            'colsample_bytree': 0.8683430739702909,
            'lambda': 3.717655605557664,
            'alpha': 2.8186169330124836e-05
            }


# --- K-Fold Training ---
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True,random_state=RANDOM_STATE)
oof_mean_preds = np.zeros(len(X))
test_mean_preds = np.zeros(len(X_test))


# THE FIX: Reload the 'grade' column for stratification as the original df was deleted.
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

# Add the other required XGBoost parameters
final_params_mean = {'objective': 'reg:squarederror', 'eval_metric': 'rmse','tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1,**best_params_mean}
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f" Mean Model - Fold {fold+1}/{N_SPLITS}...")
    model = xgb.XGBRegressor(**final_params_mean, n_estimators=2500,early_stopping_rounds=100)
    model.fit(X.iloc[train_idx], y_true.iloc[train_idx], eval_set=[(X.iloc[val_idx], y_true.iloc[val_idx])], verbose=False)
    oof_mean_preds[val_idx] = model.predict(X.iloc[val_idx])
    test_mean_preds += model.predict(X_test) / N_SPLITS

    
# --- NEW: CALCULATE AND PRINT FINAL OOF RMSE ---
final_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
print(f"\n# Mean model K-Fold training complete.")
print(f"# Final OOF RMSE for Mean Model: ${final_mean_rmse:,.2f}")
print("-" * 50)



--- STAGE 1: K-Fold Training of Mean Model ---
# Using pre-tuned, optimal hyperparameters.
 Mean Model - Fold 1/5...
 Mean Model - Fold 2/5...
 Mean Model - Fold 3/5...
 Mean Model - Fold 4/5...
 Mean Model - Fold 5/5...

# Mean model K-Fold training complete.
# Final OOF RMSE for Mean Model: $98,990.27
--------------------------------------------------


In [4]:
# =======================================================================================
#
# BLOCK 4: TRAIN THE CATBOOST MEAN MODEL
#
# =======================================================================================
print("--- STAGE 1: K-Fold Training of CatBoost Mean Model ---")
print("# Using the optimal hyperparameters found previously.")

# The best parameters you found for the CatBoost mean model
best_params_catboost = {
    'iterations': 2329,
    'learning_rate': 0.07106828663102695,
    'depth': 9,
    'l2_leaf_reg': 0.011456642952301135,
    'subsample': 0.901161247633512,
    'random_strength': 0.6138817518014036,
    'bagging_temperature': 0.017313451474763708,
    'random_seed': RANDOM_STATE,
    'verbose': 0
}

# Initialize arrays to store predictions
oof_catboost_preds = np.zeros(len(X))
test_catboost_preds = np.zeros(len(X_test))

# Setup K-Fold splits
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Training CatBoost Mean Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]
    
    model = cb.CatBoostRegressor(**best_params_catboost)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              early_stopping_rounds=100,
              verbose=0)
    
    oof_catboost_preds[val_idx] = model.predict(X_val)
    test_catboost_preds += model.predict(X_test) / N_SPLITS
    
# --- Evaluate Mean Model Performance ---
final_mean_rmse_cb = np.sqrt(mean_squared_error(y_true, oof_catboost_preds))
print("\n--- CatBoost K-Fold Training Complete & Performance Metrics ---")
print(f"CatBoost Final OOF RMSE: ${final_mean_rmse_cb:,.2f}")

--- STAGE 1: K-Fold Training of CatBoost Mean Model ---
# Using the optimal hyperparameters found previously.
  Training CatBoost Mean Model - Fold 1/5...
  Training CatBoost Mean Model - Fold 2/5...
  Training CatBoost Mean Model - Fold 3/5...
  Training CatBoost Mean Model - Fold 4/5...
  Training CatBoost Mean Model - Fold 5/5...

--- CatBoost K-Fold Training Complete & Performance Metrics ---
CatBoost Final OOF RMSE: $98,246.71


In [10]:
# =======================================================================================
#
# BLOCK 5: FIND THE OPTIMAL ENSEMBLE BLEND
#
# =======================================================================================

# ASSUMPTION: You have these variables from your previous cells:
# oof_mean_preds (XGBoost OOF predictions)
# oof_catboost_preds (CatBoost OOF predictions)
# y_true

print("\n--- Searching for the optimal blending weight for the XGBoost + CatBoost ensemble ---")

best_rmse = float('inf')
best_weight_cb = 0.5  # Default to 50/50

# We will test weights for the CatBoost model from 0 to 1, in small steps.
for w_cb in np.arange(0, 1.01, 0.05):
    w_xgb = 1.0 - w_cb
    
    # Create the weighted ensemble OOF prediction
    weighted_ensemble_oof = (oof_catboost_preds * w_cb) + (oof_mean_preds * w_xgb)
    
    # Calculate the RMSE for this specific weighting
    current_rmse = np.sqrt(mean_squared_error(y_true, weighted_ensemble_oof))
    
    print(f"Weight (CB/XGB): {w_cb:.2f}/{w_xgb:.2f}  |  RMSE: ${current_rmse:,.2f}")
    
    if current_rmse < best_rmse:
        best_rmse = current_rmse
        best_weight_cb = w_cb

print("\n" + "="*50)
print("             OPTIMAL WEIGHT SEARCH COMPLETE")
print("="*50)
print(f"Lowest Ensemble RMSE achieved: ${best_rmse:,.2f}")
print(f"Optimal Weight for CatBoost: {best_weight_cb:.2f}")
print(f"Optimal Weight for XGBoost : {1.0-best_weight_cb:.2f}")
print("="*50)
print("\nACTION: Proceeding to the full pipeline using these optimal weights.")


--- Searching for the optimal blending weight for the XGBoost + CatBoost ensemble ---
Weight (CB/XGB): 0.00/1.00  |  RMSE: $98,990.27
Weight (CB/XGB): 0.05/0.95  |  RMSE: $98,647.55
Weight (CB/XGB): 0.10/0.90  |  RMSE: $98,335.97
Weight (CB/XGB): 0.15/0.85  |  RMSE: $98,055.82
Weight (CB/XGB): 0.20/0.80  |  RMSE: $97,807.38
Weight (CB/XGB): 0.25/0.75  |  RMSE: $97,590.89
Weight (CB/XGB): 0.30/0.70  |  RMSE: $97,406.56
Weight (CB/XGB): 0.35/0.65  |  RMSE: $97,254.58
Weight (CB/XGB): 0.40/0.60  |  RMSE: $97,135.08
Weight (CB/XGB): 0.45/0.55  |  RMSE: $97,048.21
Weight (CB/XGB): 0.50/0.50  |  RMSE: $96,994.03
Weight (CB/XGB): 0.55/0.45  |  RMSE: $96,972.62
Weight (CB/XGB): 0.60/0.40  |  RMSE: $96,983.98
Weight (CB/XGB): 0.65/0.35  |  RMSE: $97,028.12
Weight (CB/XGB): 0.70/0.30  |  RMSE: $97,104.97
Weight (CB/XGB): 0.75/0.25  |  RMSE: $97,214.48
Weight (CB/XGB): 0.80/0.20  |  RMSE: $97,356.51
Weight (CB/XGB): 0.85/0.15  |  RMSE: $97,530.94
Weight (CB/XGB): 0.90/0.10  |  RMSE: $97,737.60
W

In [15]:
# =======================================================================================
#
# BLOCK 6: THE FULL OPTIMAL BLEND PIPELINE
#
# =======================================================================================

# --- PART A: Create the Optimally-Blended Ensemble Mean ---
print("\n--- PART A: Creating the optimally-blended mean predictions ---")
w_xgb_best = 1.0 - best_weight_cb
oof_ensemble_mean = (oof_catboost_preds * best_weight_cb) + (oof_mean_preds * w_xgb_best)
test_ensemble_mean = (test_catboost_preds * best_weight_cb) + (test_mean_preds * w_xgb_best)
print("Ensemble predictions created with optimal weights.")

# --- PART B: Tune a New Error Model for the Optimal Ensemble ---
print("\n--- PART B: Tuning a new error model for the optimal ensemble's errors ---")
error_target_ensemble = np.abs(y_true - oof_ensemble_mean)
X_for_error_ensemble = X.copy()
X_for_error_ensemble['mean_pred_oof'] = oof_ensemble_mean

N_OPTUNA_TRIALS = 50

def create_error_objective(X_features, y_error):
    X_train, X_val, y_train, y_val = train_test_split(X_features, y_error, test_size=0.25, random_state=RANDOM_STATE)
    def objective(trial):
        params = {
            'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist',
            'seed': RANDOM_STATE, 'n_jobs': -1,
            
            # Focused search on the most promising parameter ranges
            'eta': trial.suggest_float('eta', 0.01, 0.05, log=True),      # Narrowed: Lower learning rates were better
            'max_depth': trial.suggest_int('max_depth', 6, 9),            # Narrowed: 7 was best, so explore around it
            'subsample': trial.suggest_float('subsample', 0.8, 1.0),      # Narrowed: High subsample was consistently better
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8), # Narrowed
            'lambda': trial.suggest_float('lambda', 1e-4, 10.0, log=True),# Kept broad, but starting a bit higher
            'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),   # Shrunk: L1 reg seemed unimportant
        }
        model = xgb.XGBRegressor(**params, n_estimators=2000, early_stopping_rounds=50)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        return np.sqrt(mean_squared_error(y_val, model.predict(X_val)))
    return objective

study_error_ensemble = optuna.create_study(direction='minimize')
study_error_ensemble.optimize(create_error_objective(X_for_error_ensemble, error_target_ensemble), n_trials=N_OPTUNA_TRIALS)
best_params_error_ensemble = study_error_ensemble.best_params
print(f"\nOptimal Error Model Params Found (RMSE: ${study_error_ensemble.best_value:,.2f})")

# --- PART C: K-Fold Train the Tuned Error Model ---
print("\n--- PART C: K-Fold training the new error model ---")
final_params_error_ensemble = best_params_error_ensemble.copy()
final_params_error_ensemble.update({'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1})
oof_error_preds_ensemble = np.zeros(len(X))
test_error_preds_ensemble = np.zeros(len(X_test))
X_test_for_error_ensemble = X_test.copy()
X_test_for_error_ensemble['mean_pred_oof'] = test_ensemble_mean
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error_ensemble, grade_for_stratify)):
    print(f"  Training Ensemble Error Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X_for_error_ensemble.iloc[train_idx], X_for_error_ensemble.iloc[val_idx]
    y_train, y_val = error_target_ensemble.iloc[train_idx], error_target_ensemble.iloc[val_idx]
    model = xgb.XGBRegressor(**final_params_error_ensemble, n_estimators=2000, early_stopping_rounds=100)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_error_preds_ensemble[val_idx] = model.predict(X_val)
    test_error_preds_ensemble += model.predict(X_test_for_error_ensemble) / N_SPLITS

# --- PART D: Final Calibration and Showdown ---
print("\n--- PART D: Calibrating the final interval and getting the score ---")
OLD_BEST_SCORE = 301553.47 # Your original champion score
oof_error_final_ensemble = np.clip(oof_error_preds_ensemble, 0, None)
best_score_ensemble = float('inf')
best_a_ensemble, best_b_ensemble = 1.0, 1.0
for a in np.arange(1.90, 2.31, 0.01):
    for b in np.arange(2.10, 2.51, 0.01):
        low = oof_ensemble_mean - oof_error_final_ensemble * a
        high = oof_ensemble_mean + oof_error_final_ensemble * b
        score = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA)
        if score < best_score_ensemble:
            best_score_ensemble = score
            best_a_ensemble, best_b_ensemble = a, b

print("\n" + "="*60)
print("             THE OPTIMAL BLEND PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Original XGBoost Pipeline Final Score: {OLD_BEST_SCORE:,.2f}")
print(f"New OPTIMAL BLEND Pipeline Final Score: {best_score_ensemble:,.2f}")
print(f"  (Using optimal multipliers a={best_a_ensemble:.2f}, b={best_b_ensemble:.2f})")

if best_score_ensemble < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The optimally blended pipeline is the new champion!")
else:
    print("\nCONCLUSION: Close! The original XGBoost pipeline remains superior.")

[I 2025-07-16 17:57:08,909] A new study created in memory with name: no-name-f893a556-ad44-4551-a6fe-f503410e1379



--- PART A: Creating the optimally-blended mean predictions ---
Ensemble predictions created with optimal weights.

--- PART B: Tuning a new error model for the optimal ensemble's errors ---


[I 2025-07-16 17:57:15,441] Trial 0 finished with value: 61730.34911611204 and parameters: {'eta': 0.02127174836607627, 'max_depth': 8, 'subsample': 0.8124862633893231, 'colsample_bytree': 0.578891278214649, 'lambda': 0.0006724033071260583, 'alpha': 1.9273002965399258e-08}. Best is trial 0 with value: 61730.34911611204.
[I 2025-07-16 17:57:23,043] Trial 1 finished with value: 61943.59900363203 and parameters: {'eta': 0.01781619213010289, 'max_depth': 6, 'subsample': 0.9523162874905396, 'colsample_bytree': 0.525034966817347, 'lambda': 0.0002077576989237827, 'alpha': 1.252023495713962e-07}. Best is trial 0 with value: 61730.34911611204.
[I 2025-07-16 17:57:31,142] Trial 2 finished with value: 61741.118329871286 and parameters: {'eta': 0.021309507537327046, 'max_depth': 7, 'subsample': 0.9177890490882014, 'colsample_bytree': 0.6181420494816725, 'lambda': 1.3255350934623418, 'alpha': 2.2483958863091568e-07}. Best is trial 0 with value: 61730.34911611204.
[I 2025-07-16 17:57:51,108] Trial 3


Optimal Error Model Params Found (RMSE: $61,590.88)

--- PART C: K-Fold training the new error model ---
  Training Ensemble Error Model - Fold 1/5...
  Training Ensemble Error Model - Fold 2/5...
  Training Ensemble Error Model - Fold 3/5...
  Training Ensemble Error Model - Fold 4/5...
  Training Ensemble Error Model - Fold 5/5...

--- PART D: Calibrating the final interval and getting the score ---

             THE OPTIMAL BLEND PIPELINE: FINAL SHOWDOWN
Original XGBoost Pipeline Final Score: 301,553.47
New OPTIMAL BLEND Pipeline Final Score: 297,707.05
  (Using optimal multipliers a=1.95, b=2.18)

CONCLUSION: VICTORY! The optimally blended pipeline is the new champion!


In [14]:
# =======================================================================================
#
# BLOCK 7: CREATE FINAL SUBMISSION
#
# =======================================================================================
OLD_BEST_SCORE = 301553.47 # Your champion score from winner_v1_301
if best_score_ensemble < OLD_BEST_SCORE:
    print("\n--- Creating the new champion submission file... ---")
    correct_test_ids = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])['id']
    test_error_final_ensemble = np.clip(test_error_preds_ensemble, 0, None)
    final_lower = test_ensemble_mean - test_error_final_ensemble * best_a_ensemble
    final_upper = test_ensemble_mean + test_error_final_ensemble * best_b_ensemble
    final_upper = np.maximum(final_lower, final_upper)
    submission_df = pd.DataFrame({'id': correct_test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
    submission_filename = 'submission_optimal_blend_v1_aa.csv'
    submission_df.to_csv(submission_filename, index=False)
    print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
    display(submission_df.head())
else:
    print("\nNew submission file not created as the score was not an improvement.")


--- Creating the new champion submission file... ---

'submission_optimal_blend_v1_aa.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,833141.717679,1.059718e+06
1,200001,581060.130551,7.967188e+05
2,200002,446592.038757,6.495708e+05
3,200003,286773.396554,4.153863e+05
4,200004,288641.590469,7.625976e+05


In [7]:
# =======================================================================================
#
# BLOCK 7: STAGE 4 - FINAL CALIBRATION AND ENSEMBLE SHOWDOWN
#
# =======================================================================================
print("\n" + "="*60)
print("             THE ENSEMBLE PIPELINE: FINAL SHOWDOWN")
print("="*60)

OLD_BEST_SCORE = 301553.47 # Your champion score from winner_v1_301

oof_error_final_ensemble = np.clip(oof_error_preds_ensemble, 0, None)
best_score_ensemble = float('inf')
best_a_ensemble, best_b_ensemble = 1.0, 1.0

print("Starting final calibration grid search for the ENSEMBLE pipeline...")
for a in np.arange(1.90, 2.31, 0.01):
    for b in np.arange(2.10, 2.51, 0.01):
        low = oof_ensemble_mean - oof_error_final_ensemble * a
        high = oof_ensemble_mean + oof_error_final_ensemble * b
        score = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA)
        if score < best_score_ensemble:
            best_score_ensemble = score
            best_a_ensemble, best_b_ensemble = a, b

print("\n--- FINAL RESULTS ---")
print(f"Original XGBoost Pipeline Final Score: {OLD_BEST_SCORE:,.2f}")
print(f"New ENSEMBLE Pipeline Final Score    : {best_score_ensemble:,.2f}")
print(f"  (Using optimal multipliers a={best_a_ensemble:.2f}, b={best_b_ensemble:.2f})")

if best_score_ensemble < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The Ensemble-based pipeline is the new champion!")
else:
    print("\nCONCLUSION: Close! The original XGBoost pipeline remains superior.")


             THE ENSEMBLE PIPELINE: FINAL SHOWDOWN
Starting final calibration grid search for the ENSEMBLE pipeline...

--- FINAL RESULTS ---
Original XGBoost Pipeline Final Score: 301,553.47
New ENSEMBLE Pipeline Final Score    : 298,168.87
  (Using optimal multipliers a=1.95, b=2.17)

CONCLUSION: VICTORY! The Ensemble-based pipeline is the new champion!


In [9]:
# =======================================================================================
#
# BLOCK 8: STAGE 5 - CREATE FINAL SUBMISSION (CORRECTED)
#
# =======================================================================================
if best_score_ensemble < OLD_BEST_SCORE:
    print("\n--- Creating the new champion submission file... ---")
    
    # --- THE FIX IS HERE ---
    # Ignore any previous 'test_ids' variable.
    # Load the correct, original IDs directly from the test.csv file.
    # We use `usecols` to make this very fast and memory-efficient.
    print("Loading original IDs from test.csv...")
    correct_test_ids = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])['id']
    
    # The rest of the prediction calculations are correct as they are already in the right order.
    test_error_final_ensemble = np.clip(test_error_preds_ensemble, 0, None)
    final_lower = test_ensemble_mean - test_error_final_ensemble * best_a_ensemble
    final_upper = test_ensemble_mean + test_error_final_ensemble * best_b_ensemble
    final_upper = np.maximum(final_lower, final_upper)
    
    # Create the submission DataFrame using the CORRECT IDs.
    submission_df = pd.DataFrame({'id': correct_test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
    submission_filename = 'submission_ensemble_pipeline_v1_CORRECT_IDS.csv'
    submission_df.to_csv(submission_filename, index=False)

    print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
    display(submission_df.head())
else:
    print("\nNew submission file not created as the score was not an improvement.")


--- Creating the new champion submission file... ---
Loading original IDs from test.csv...

'submission_ensemble_pipeline_v1_CORRECT_IDS.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,834335.404947,1.058136e+06
1,200001,584252.156280,7.980737e+05
2,200002,448058.425868,6.491074e+05
3,200003,286412.686713,4.143234e+05
4,200004,260368.980318,7.972589e+05
